# **JOINS**

In [0]:
from pyspark.sql.functions import *

In [0]:
st_df = spark.read.csv('/Volumes/workspace/default/apacha-data/join1.csv', header=True, inferSchema=True)
st_df.show()

In [0]:
enroll_df = spark.read.csv('/Volumes/workspace/default/apacha-data/join2.csv', header=True, inferSchema=True)
enroll_df.show()

In [0]:
st_df.join(enroll_df, st_df.student_id == enroll_df.student_id, 'inner').show()

In [0]:
st_df.join(enroll_df,st_df.student_id==enroll_df.student_id, 'left_outer').show()

In [0]:
st_df.join(enroll_df,st_df.student_id==enroll_df.student_id, 'right_outer').show()

In [0]:
result = st_df.join(enroll_df, "student_id", "fullouter") \
              .orderBy(col("student_id").asc_nulls_last())

result.show()

In [0]:
st_df.join(enroll_df,st_df.student_id==enroll_df.student_id, 'left_semi').show()

In [0]:
st_df.join(enroll_df,st_df.student_id==enroll_df.student_id, 'left_anti').show()

# **Union & Union all**

In [0]:
import pyspark
from pyspark.sql import SparkSession

data1 = [
  ("Ali","Khan","Ahmed","50001","M",3500),
  ("Sara","","Iqbal","50002","F",4200),
  ("Usman","Raza","Sheikh","50003","M",3900),
  ("Ayesha","Noor","Malik","50004","F",4100),
  ("Bilal","","Hussain","50005","M",3000)
]

columns1 = ["firstname","middlename","lastname","id","gender","salary"]
df1 = spark.createDataFrame(data=data1, schema = columns1)
df1.show()

In [0]:
data2 = [
  ("Ali","Khan","Ahmed","50001","M",3500),   # duplicate row
  ("Hina","Fatima","Qureshi","50006","F",4500),
  ("Zain","","Ali","50007","M",3800),
  ("Sara","","Iqbal","50002","F",4200),     # duplicate row
  ("Hamza","Tariq","Butt","50008","M",3600)
]

columns2 = ["firstname","middlename","lastname","id","gender","salary"]
df2 = spark.createDataFrame(data=data2, schema = columns2)
df2.show()

In [0]:
df1.printSchema()

In [0]:
df2.printSchema()

In [0]:
df1.union(df2).display()

In [0]:
display(df1.unionAll(df2))

In [0]:
df1.union(df2).distinct().display()

# **fill and fillna**

In [0]:
df3 = spark.read.csv('/Volumes/workspace/default/apacha-data/fill.csv', header=True, inferSchema=True)
df3.show()

In [0]:
df3.na.fill("unknown").display()

In [0]:
df3.fillna(0).display()

In [0]:
df3.na.fill("unknown",["name"]).display()

In [0]:
df3.fillna("Ayesha",["name"]).fillna(4000,["salary"]).na.fill("Hyderabad",["city"]).na.fill("M",["gender"]).display()

## **When**

In [0]:
df3 = df3.withColumn("salary",
                   when(col("salary") == -1, None)
                   .otherwise(col("salary")))

df3.display()

In [0]:
df_updated = (
    df3
    .withColumn(
        "age",
        when(col("name") == "Sara", 27)
        .when(col("name") == "Fatima", 23)
        .when(col("name") == "Usman", 19)
        .when(col("name") == "Hassan", 34)
        .otherwise(col("age"))
    )
    .withColumn(
        "name",
        when(col("age") == 22, "Ayesha")
        .when(col("age") == 29, "Amaan")
        .otherwise(col("name"))
    )
)

df_updated.show()

# **StructType & StructField**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

data = [(1, "Amaan", "pakistan", 5.76), (2, "Bilal", "India", 5.96), (3, "Fatima", "USA", 5.56), (4, "Hassan", "China", 5.66)]

schema = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("height", FloatType(), True)
])

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

# **Pivot & unpivot**

In [0]:
data = [("banana", 1000, "USA"), ("orange", 3100, "USA"), ("carrot", 6200, "China"), ("banana", 2000, "USA"), ("banana", 6200, "China"), ("orange", 7300, "China"), ("banana", 7600, "China"), ("carrot", 8200, "China"), ("carrot", 4000, "USA"), ("carrot", 3000, "India"), ("orange", 2000, "India"), ("orange", 4000, "India"), ("carrot", 5000, "India")]

columns = ["product", "price", "country"]
df = spark.createDataFrame(data = data, schema = columns)
df.show()

In [0]:
df_pivot = df.groupBy("product").pivot("country").sum("price")
df_pivot.show()

In [0]:
df_unpivot = df_pivot
df_unpivot.show()

In [0]:
df_unpivot.select("product",expr("stack(3,'USA',USA,'India',India,'China',China) as (country,total)")).show()